Four filters in sequence. A document has to survive all four to get written out.

1. Length — drop anything under 200 chars or over 100k. Short docs are nav menus and cookie banners; giant ones are usually scraped junk or concatenated garbage.

2. Exact duplicate — hash the whole document, keep a set of hashes seen. Same text twice → drop the second. Web crawls are full of identical pages on mirror domains.

3. Script check — for Telugu, count what fraction of the letters are in the Telugu Unicode block (\u0c00–\u0c7f). Below 70% → drop. This catches English pages that FineWeb mislabeled as Telugu, and romanized Telugu, which you don't want mixed with native script.

4. Near-duplicate — the only non-obvious one. This catches documents that are almost the same: same article with a different headline, same product page with one field changed. Exact hashing misses those completely, because one changed character gives a totally different hash.

How step 4 works:

Cut the document into overlapping 5-word chunks. "the cat sat on the mat" → "the cat sat on the", "cat sat on the mat".
Hash every chunk. Take the minimum hash. That minimum is a fingerprint of the document — and here's the trick: two documents that share most of their chunks will usually share the same minimum, because the same chunk wins in both.
Do that 32 times with 32 different hash functions (the PERM affine permutations) → 32 fingerprints.
Group them into 8 bands of 4. If a new document matches an old one on 2 or more bands, it's a near-duplicate → drop.

Why bands instead of comparing all 32: comparing every document to every other is 17M² comparisons. Banding turns it into 8 dictionary lookups. That's the whole reason this technique exists.

Verified on synthetic data: near-dupes shared 6/8 bands, unrelated documents shared 0/8.

Why any of this matters for you: duplicated text is the fastest way to make a model memorize instead of learn, and it inflates your token count so you think you have more Telugu than you do. drop_near and drop_exact in the output tell you how much of your corpus was fake volume.

In [1]:
import os, random, unicodedata
from hashlib import blake2b
from collections import Counter

In [2]:
ROOT = "E:/Project_SLM"
NGRAM, BANDS, ROWS = 5, 8, 4
MIN_CHARS, MAX_CHARS = 200, 100_000 ##point1
MASK = (1 << 61) - 1
random.seed(0)
PERM = [(random.getrandbits(60) | 1, random.getrandbits(60)) for _ in range(BANDS * ROWS)]
SCRIPTS = {
    "te": (lambda c: "\u0c00" <= c <= "\u0c7f", 0.70),
    "de": (lambda c: (c.isascii() and c.isalpha()) or c in "äöüÄÖÜß", 0.85),
    "en": (lambda c: c.isascii(), 0.95),
}

In [3]:
import urllib.request, os
p = "E:/Project_SLM/lid.176.bin"
if not os.path.exists(p):
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin", p)
print(os.path.getsize(p)/1e6, "MB")   # expect ~126

import fasttext
LID = fasttext.load_model(p)
print(LID.predict("Das ist ein deutscher Satz zum Testen.", k=2))
print(LID.predict("This is an english sentence for testing.", k=2))

131.266198 MB
(('__label__de',), array([1.00003779]))
(('__label__en', '__label__te'), array([0.97383201, 0.00205393]))


In [4]:
def signature(doc):
    w = doc.split()
    grams = {" ".join(w[i:i+NGRAM]) for i in range(len(w)-NGRAM+1)} or {doc}
    base = [int.from_bytes(blake2b(g.encode(), digest_size=8).digest(), "big") & MASK for g in grams]
    mins = [min((a*h+b) & MASK for h in base) for a, b in PERM]
    return [b"".join(m.to_bytes(8,"big") for m in mins[i*ROWS:(i+1)*ROWS]) for i in range(BANDS)]

In [5]:
def clean(lang):
    test, min_ratio = SCRIPTS[lang]
    seen, bands, s = set(), [set() for _ in range(BANDS)], Counter()
    os.makedirs(f"{ROOT}/clean", exist_ok=True)
    with open(f"{ROOT}/raw/{lang}.txt", encoding="utf-8") as fin, \
         open(f"{ROOT}/clean/{lang}.txt", "w", encoding="utf-8") as fout:
        for line in fin:
            s["in"] += 1
            doc = unicodedata.normalize("NFC", line.strip())
            if not (MIN_CHARS <= len(doc) <= MAX_CHARS):
                s["drop_len"] += 1; continue
            h = blake2b(doc.encode(), digest_size=16).digest()
            if h in seen:
                s["drop_exact"] += 1; continue
            seen.add(h)
            letters = [c for c in doc if c.isalpha()]
            if not letters or sum(test(c) for c in letters)/len(letters) < min_ratio:
                s["drop_script"] += 1; continue
            sig = signature(doc)
            if sum(b in bands[i] for i, b in enumerate(sig)) >= 2:
                s["drop_near"] += 1; continue
            for i, b in enumerate(sig): bands[i].add(b)
            fout.write(doc + "\n"); s["kept"] += 1; s["bytes"] += len(doc.encode())
            if s["in"] % 200_000 == 0:
                print(f"  {s['in']:,} read, {s['kept']:,} kept, {s['bytes']/1e9:.2f} GB", flush=True)
    print(f"\n{lang}: in={s['in']:,} kept={s['kept']:,} ({s['kept']/max(s['in'],1):.1%}) {s['bytes']/1e9:.2f} GB")
    for k in ("drop_len","drop_exact","drop_script","drop_near"): print(f"   {k:<12}{s[k]:>10,}")
    return s



In [6]:
# stats = clean("te")

In [ ]:
# from collections import Counter
# from itertools import islice
# with open("E:/Project_SLM/clean/de.txt", encoding="utf-8") as f:
#     sample = [l.strip() for l in islice(f, 500)]

# print(Counter(LID.predict(s[:1000])[0][0] for s in sample))

Counter({'__label__de': 499, '__label__en': 1})


In [6]:
stats_de = clean("de")

  200,000 read, 199,421 kept, 0.62 GB
  400,000 read, 398,639 kept, 1.15 GB
  600,000 read, 598,247 kept, 1.70 GB
  800,000 read, 797,804 kept, 2.28 GB
  1,000,000 read, 997,273 kept, 2.88 GB
  1,200,000 read, 1,196,706 kept, 3.54 GB
  1,400,000 read, 1,396,003 kept, 4.19 GB
  1,600,000 read, 1,595,056 kept, 4.76 GB
  1,800,000 read, 1,794,393 kept, 5.31 GB
  2,000,000 read, 1,993,760 kept, 5.90 GB
  2,200,000 read, 2,193,034 kept, 6.51 GB
  2,400,000 read, 2,392,297 kept, 7.18 GB
  2,600,000 read, 2,591,313 kept, 7.81 GB
  2,800,000 read, 2,790,424 kept, 8.36 GB
  3,000,000 read, 2,989,527 kept, 8.95 GB
  3,200,000 read, 3,188,581 kept, 9.58 GB
  3,400,000 read, 3,387,719 kept, 10.24 GB
  3,600,000 read, 3,586,400 kept, 10.82 GB
  3,800,000 read, 3,785,356 kept, 11.36 GB
  4,000,000 read, 3,984,297 kept, 11.92 GB
  4,200,000 read, 4,183,243 kept, 12.51 GB
  4,400,000 read, 4,382,077 kept, 13.13 GB
  4,600,000 read, 4,580,987 kept, 13.79 GB

de: in=4,721,706 kept=4,702,096 (99.6%) 14.1

In [8]:
stats_en = clean("en")

  200,000 read, 199,324 kept, 0.92 GB
  400,000 read, 398,182 kept, 1.84 GB
  600,000 read, 596,592 kept, 2.77 GB
  800,000 read, 794,190 kept, 3.68 GB
  1,000,000 read, 991,460 kept, 4.59 GB
  1,200,000 read, 1,187,956 kept, 5.50 GB
  1,400,000 read, 1,384,137 kept, 6.41 GB
  1,600,000 read, 1,579,655 kept, 7.31 GB
  1,800,000 read, 1,774,860 kept, 8.21 GB
  2,000,000 read, 1,969,327 kept, 9.12 GB
  2,200,000 read, 2,163,355 kept, 10.02 GB
  2,400,000 read, 2,356,905 kept, 10.91 GB
  2,600,000 read, 2,549,715 kept, 11.80 GB
  2,800,000 read, 2,742,329 kept, 12.68 GB
  3,000,000 read, 2,934,609 kept, 13.58 GB
  3,200,000 read, 3,125,534 kept, 14.44 GB
  3,400,000 read, 3,315,102 kept, 15.29 GB
  3,600,000 read, 3,504,331 kept, 16.13 GB

en: in=3,761,016 kept=3,655,812 (97.2%) 16.81 GB
   drop_len         3,204
   drop_exact      60,582
   drop_script      5,953
   drop_near       35,465


In [9]:
import os, unicodedata
from datasets import load_dataset

CAP = 2000 * 1024**2          # 2 GB

os.makedirs(f"{ROOT}/raw", exist_ok=True)
ds = load_dataset("ai4bharat/sangraha", data_dir="verified/tel",
                  split="train", streaming=True)

written = 0
with open(f"{ROOT}/raw/te_sangraha.txt", "w", encoding="utf-8") as f:
    for row in ds:
        doc = row.get("text") or row.get("content") or ""
        doc = unicodedata.normalize("NFC", doc).strip().replace("\n", " ")
        if len(doc) < 200:
            continue
        f.write(doc + "\n")
        written += len(doc.encode("utf-8"))
        if written >= CAP:
            break
print(f"done: {written/1e9:.2f} GB -> {ROOT}/raw/te_sangraha.txt")


README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

c:\Users\nikhi\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikhi\.cache\huggingface\hub\datasets--ai4bharat--sangraha. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

done: 2.10 GB -> E:/Project_SLM/raw/te_sangraha.txt


In [10]:
def clean_multi(lang, infiles, outfile):
    test, min_ratio = SCRIPTS[lang]
    seen, bands, s = set(), [set() for _ in range(BANDS)], Counter()
    with open(outfile, "w", encoding="utf-8") as fout:
        for path in infiles:                      # <- shared seen/bands across files
            print(f"--- {path}", flush=True)
            for line in open(path, encoding="utf-8"):
                s["in"] += 1
                doc = unicodedata.normalize("NFC", line.strip())
                if not (MIN_CHARS <= len(doc) <= MAX_CHARS):
                    s["drop_len"] += 1; continue
                h = blake2b(doc.encode(), digest_size=16).digest()
                if h in seen:
                    s["drop_exact"] += 1; continue
                seen.add(h)
                letters = [c for c in doc if c.isalpha()]
                if not letters or sum(test(c) for c in letters)/len(letters) < min_ratio:
                    s["drop_script"] += 1; continue
                sig = signature(doc)
                if sum(b in bands[i] for i, b in enumerate(sig)) >= 2:
                    s["drop_near"] += 1; continue
                for i, b in enumerate(sig): bands[i].add(b)
                fout.write(doc + "\n"); s["kept"] += 1; s["bytes"] += len(doc.encode())
                if s["in"] % 200_000 == 0:
                    print(f"  {s['in']:,} read, {s['kept']:,} kept, {s['bytes']/1e9:.2f} GB", flush=True)
    print(f"\n{lang}: in={s['in']:,} kept={s['kept']:,} ({s['kept']/max(s['in'],1):.1%}) {s['bytes']/1e9:.2f} GB")
    for k in ("drop_len","drop_exact","drop_script","drop_near"): print(f"   {k:<12}{s[k]:>10,}")
    return s

stats_te = clean_multi("te",
    ["E:/Project_SLM/raw/te.txt", "E:/Project_SLM/raw/te_sangraha.txt"],
    "E:/Project_SLM/clean/te.txt")

--- E:/Project_SLM/raw/te.txt
  200,000 read, 192,692 kept, 1.75 GB
  400,000 read, 384,922 kept, 3.37 GB
  600,000 read, 576,637 kept, 4.92 GB
  800,000 read, 767,725 kept, 6.44 GB
--- E:/Project_SLM/raw/te_sangraha.txt
  1,000,000 read, 957,025 kept, 7.62 GB
  1,200,000 read, 1,145,293 kept, 8.62 GB

te: in=1,266,591 kept=1,208,024 (95.4%) 8.95 GB
   drop_len           225
   drop_exact         841
   drop_script     36,930
   drop_near       20,571


In [1]:
import os
ROOT = "E:/Project_SLM"
os.makedirs(f"{ROOT}/heldout", exist_ok=True)

def split_holdout(lang, mb=50, every=37):
    """Take every Nth doc into heldout, rest into train. Not the head."""
    cap, held = mb * 1024**2, 0
    src = f"{ROOT}/clean/{lang}.txt"
    with open(src, encoding="utf-8") as f, \
         open(f"{ROOT}/heldout/{lang}.txt", "w", encoding="utf-8") as h, \
         open(f"{ROOT}/train/{lang}.txt", "w", encoding="utf-8") as t:
        for i, line in enumerate(f):
            if i % every == 0 and held < cap:
                h.write(line); held += len(line.encode())
            else:
                t.write(line)
    print(f"{lang}: heldout {held/1e6:.0f} MB")

os.makedirs(f"{ROOT}/train", exist_ok=True)
for l in ("te", "de", "en"):
    split_holdout(l)

te: heldout 52 MB
de: heldout 52 MB
en: heldout 52 MB
